# LightRAG (2024)
[[paper]](https://arxiv.org/abs/2410.05079)<br>
LightRAG = Lightweight Retrieval-Augmented Generation

__LightRAG__ — это фреймворк для построения систем Retrieval-Augmented Generation, использующий графовую структуру индекса и двухэтапную стратегию поиска (Dual-level Retrieval). Он позволяет эффективно отвечать как на узкоспециализированные запросы, так и на вопросы, требующие агрегации знаний по всему корпусу документов, при этом значительно снижая вычислительные затраты по сравнению с существующими графовыми подходами.

__Постановка задачи__<br>
Улучшение качества генерации ответов LLM путем предоставления релевантного контекста из внешней базы знаний. Основная цель — обеспечить высокую точность поиска (Retrieval) для запросов разной степени абстракции: от конкретных фактов до глобальных тем.

__Мотивация__<br>
Стандартные системы RAG на основе векторного поиска (Vector RAG) хорошо находят фрагменты текста по семантическому сходству, но плохо справляются с задачами, требующими сопоставления разрозненной информации из разных частей документов (Multi-hop Reasoning) или понимания общей структуры знаний. Существующие графовые решения (GraphRAG) решают эту проблему через индексацию сущностей и связей, но они крайне дороги в эксплуатации и медленны, так как требуют предварительного суммаризирования всех узлов и сообществ в графе.

__Существующие подходы__<br>
На момент появления LightRAG основными альтернативами были:
- Vector RAG (2020): поиск по Cosine Similarity эмбеддингов. Не видит структурных связей между документами и "слеп" к глобальным темам.
- GraphRAG (Microsoft, 2024): использует Community Detection для группировки узлов графа и генерирует текстовые суммаризации для каждого сообщества. Это позволяет отвечать на глобальные вопросы, но создание индекса требует огромного количества токенов LLM (высокая стоимость), а инференс замедлен из-за необходимости читать тяжелые суммаризации.

__Идея__<br>
Разработчики предложили объединить графовую структуру с механизмом Dual-level Retrieval. Вместо дорогостоящего вычисления суммаризаций сообществ (как в Microsoft GraphRAG), модель динамически извлекает контекст на двух уровнях абстракции: Low-level (конкретные сущности и их свойства) и High-level (абстрактные темы и общие концепции). Это достигается за счет специальной структуры графа, где текстовые чанки, сущности и отношения связаны в единую сеть.

__Архитектура__<br>
Система состоит из трех основных компонентов:
1. Graph-based Text Indexing: Процесс построения графа. Документы разбиваются на чанки. Из каждого чанка LLM извлекает сущности (Nodes) и отношения (Edges). В отличие от классических графов знаний, здесь каждый чанк текста сохраняется как отдельный тип узла, связанный с извлеченными из него сущностями.
2. Dual-level Retrieval: Система поиска, работающая в двух режимах одновременно.
3. Fragment-wise Generator: Модуль формирования итогового ответа на основе собранных графовых путей и чанков текста.

__Алгоритм обучения__<br>
LightRAG не требует дообучения (fine-tuning) весов основной LLM. Основной "обучающий" процесс заключается в построении индекса:
1. Extraction: LLM проходит по тексту и извлекает сущности и триплеты (субъект-предикат-объект).
2. Deduplication: Идентичные сущности, найденные в разных частях корпуса, объединяются в один узел, создавая мостики между документами.
3. Embedding: Для каждого узла и ребра вычисляются векторы (эмбеддинги) для быстрого векторного поиска.

__Алгоритм инференса__<br>
Процесс обработки запроса пользователя:
1. Query Keyword Extraction: LLM анализирует запрос и генерирует два набора ключевых слов: конкретные (для Low-level поиска) и абстрактные (для High-level поиска).
2. Dual-level Matching:
   - Low-level: ищутся сущности в графе, которые максимально близки к конкретным ключевым словам. Это позволяет найти специфические факты.
   - High-level: ищутся высокоуровневые темы и связанные с ними группы сущностей.
3. Graph Traversal: Система собирает "подграф" из найденных узлов, ребер и связанных с ними исходных текстовых чанков.
4. Response Generation: Весь собранный контекст (триплеты + текст) подается в LLM для генерации финального ответа.

__Результаты__<br>
Эффективность проверялась на датасетах MultiHop-RAG и наборах данных из разных областей (Agriculture, CS, Legal):
- Стоимость индексации: LightRAG требует в 100+ раз меньше токенов по сравнению с Microsoft GraphRAG за счет отказа от предварительной суммаризации сообществ.
- Качество ответов: На комплексных запросах (Comprehensive Queries) LightRAG превзошел Vector RAG по метрике Recall на 20-30%.
- Скорость: Время подготовки ответа сократилось в разы по сравнению с GraphRAG, так как объем извлекаемого контекста жестко ограничен релевантным подграфом, а не огромными суммаризациями сообществ.

__Пример реализации__<br>
Ниже представлен пример инициализации и использования библиотеки:
```python
from lightrag import LightRAG, QueryParam

# Инициализация с указанием рабочей директории и модели
rag = LightRAG(working_dir="./index_db", llm_model_func=gpt_4o_complete)

# Индексация текста (автоматически строит граф сущностей)
with open("corpus.txt") as f:
    rag.insert(f.read())

# Выполнение глобального (High-level) запроса
# Метод 'hybrid' объединяет low-level и high-level поиск
print(rag.query(
    "Какие основные технологические тренды описаны в документах?", 
    param=QueryParam(mode="hybrid")
))
```
Метод `insert()` выполняет декомпозицию текста и экстракцию графа, а параметр `mode="hybrid"` в методе `query()` активирует двухслойный поиск по ключевым словам.

## 📝 Критический анализ

```markdown
# LightRAG (2024)
---
[[paper]](https://arxiv.org/abs/2410.05079)<br>
LightRAG = Lightweight Retrieval-Augmented Generation

__LightRAG__ — фреймворк для Retrieval-Augmented Generation, использующий графовую структуру индекса и стратегию Dual-level Retrieval. Он эффективно отвечает на узкоспециализированные запросы и вопросы, требующие агрегации знаний, снижая вычислительные затраты по сравнению с существующими графовыми подходами.

__Постановка задачи__<br>
Улучшение генерации ответов LLM через предоставление релевантного контекста из внешней базы знаний. Цель — высокая точность поиска для запросов разной абстракции.

__Мотивация__<br>
Стандартные системы RAG на основе Vector RAG хорошо находят текст по семантическому сходству, но плохо справляются с Multi-hop Reasoning. GraphRAG решает это через индексацию сущностей, но требует много ресурсов.

__Существующие подходы__<br>
- Vector RAG (2020): поиск по Cosine Similarity, не видит структурных связей.
- GraphRAG (Microsoft, 2024): использует Community Detection, но требует много токенов и медленен.

__Идея__<br>
Объединение графовой структуры с Dual-level Retrieval. Вместо суммаризаций, модель динамически извлекает контекст на двух уровнях: Low-level и High-level.

__Архитектура__<br>
1. Graph-based Text Indexing: Документы разбиваются на чанки, из которых LLM извлекает сущности и отношения.
2. Dual-level Retrieval: Поиск в двух режимах.
3. Fragment-wise Generator: Формирование ответа на основе графовых путей и чанков.

__Алгоритм обучения__<br>
LightRAG не требует дообучения LLM. Основной процесс — построение индекса:
1. Extraction: Извлечение сущностей и триплетов.
2. Deduplication: Объединение идентичных сущностей.
3. Embedding: Вычисление эмбеддингов для узлов и ребер.

__Алгоритм инференса__<br>
1. Query Keyword Extraction: Генерация ключевых слов для Low-level и High-level поиска.
2. Dual-level Matching: Поиск сущностей и тем.
3. Graph Traversal: Сбор "подграфа".
4. Response Generation: Генерация ответа LLM.

<img src="img/img.png" width=500>

__Результаты__<br>
- Индексация: LightRAG требует в 100+ раз меньше токенов, чем Microsoft GraphRAG.
- Качество: Превосходит Vector RAG по Recall на 20-30% на комплексных запросах.
- Скорость: Ответы готовятся быстрее, чем в GraphRAG.

__Пример реализации__<br>
```python
from lightrag import LightRAG, QueryParam

rag = LightRAG(working_dir="./index_db", llm_model_func=gpt_4o_complete)

with open("corpus.txt") as f:
    rag.insert(f.read())

print(rag.query(
    "Какие основные технологические тренды описаны в документах?", 
    param=QueryParam(mode="hybrid")
))
```
Метод `insert()` выполняет декомпозицию текста и экстракцию графа, а `mode="hybrid"` активирует двухслойный поиск.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример реализации LightRAG с использованием гипотетической библиотеки lightrag

# Импортируем необходимые классы из библиотеки
from lightrag import LightRAG, QueryParam

# Инициализация LightRAG с указанием рабочей директории и функции модели LLM
# Здесь gpt_4o_complete - это функция, которая обрабатывает запросы с помощью LLM
rag = LightRAG(working_dir="./index_db", llm_model_func=gpt_4o_complete)

# Индексация текста из файла corpus.txt
# Метод insert() автоматически строит граф сущностей из текста
with open("corpus.txt") as f:
    rag.insert(f.read())

# Выполнение глобального (High-level) запроса
# Метод query() с параметром mode="hybrid" активирует двухуровневый поиск
result = rag.query(
    "Какие основные технологические тренды описаны в документах?", 
    param=QueryParam(mode="hybrid")
)

# Вывод результата запроса
print(result)

# Пример выполнения конкретного (Low-level) запроса
# Здесь мы ищем специфическую информацию, например, о конкретной технологии
specific_result = rag.query(
    "Что известно о технологии блокчейн в документах?", 
    param=QueryParam(mode="low-level")
)

# Вывод результата конкретного запроса
print(specific_result)

# Пример выполнения абстрактного (High-level) запроса
# Здесь мы ищем общие темы или концепции
abstract_result = rag.query(
    "Какие концепции связаны с устойчивым развитием?", 
    param=QueryParam(mode="high-level")
)

# Вывод результата абстрактного запроса
print(abstract_result)
```

### Комментарии к коду:

1. **Инициализация LightRAG**: 
   - `LightRAG` инициализируется с рабочей директорией для хранения индекса и функцией модели LLM (`gpt_4o_complete`), которая будет использоваться для обработки запросов.

2. **Индексация текста**:
   - Метод `insert()` принимает текст и автоматически разбивает его на чанки, извлекает сущности и строит графовую структуру. Это позволяет эффективно организовать данные для последующего поиска.

3. **Запросы**:
   - **Гибридный запрос** (`mode="hybrid"`): Используется для поиска, который объединяет конкретные и абстрактные уровни, что позволяет находить как специфические факты, так и общие темы.
   - **Конкретный запрос** (`mode="low-level"`): Ориентирован на поиск специфических фактов или сущностей.
   - **Абстрактный запрос** (`mode="high-level"`): Ориентирован на поиск общих тем или концепций.

4. **Вывод результатов**:
   - Результаты каждого запроса выводятся с помощью `print()`, что позволяет увидеть, как LightRAG обрабатывает и возвращает информацию в зависимости от типа запроса.